# Minimum-Sensor Preprocessing & 2-Class Anxiety Classification: EDA + TEMP Only (Alkurdi Alignment)

This notebook implements the **6th try (2-Class)** pipeline with **Wrist EDA and Wrist Skin Temperature (TEMP) only** for subjective state anxiety classification on the WESAD dataset, incorporating methodological alignments with Alkurdi et al.'s reference pipeline (`_forked`):

1. **Sensor Pair Configuration (EDA + TEMP):**
   - Combines the primary autonomic sympathetic biomarker (**EDA**, 14 features) with peripheral thermal vasomotor tone (**TEMP**, 6 features).
   - All mechanical motion / Accelerometer features (**ACC**), optical **BVP**, and **chest harness sensors** are completely excluded to assess whether autonomic physiology alone can detect state anxiety in the total absence of physical movement data.
   - Generates **20 predictive features** (14 wrist EDA including phasic SCR / tonic SCL decomposition, and 6 wrist TEMP features).

2. **2-Class Baseline-Relative STAI Classification Task:**
   - Baseline protocol serves as the participant's resting reference condition.
   - Score difference: $\Delta = \text{condition\_stai\_sum} - \text{baseline\_stai\_sum}$
   - **Class 1 (Anxiety Increased):** $\Delta > 0$
   - **Class 0 (No Anxiety Increase):** $\Delta \le 0$
   - Zero-inflation completely eliminated (100% of participants exhibit $\Delta > 0$ during TSST).

3. **Sensor Clock Synchronization (Alkurdi Style):**
   - Follows Alkurdi's `pd.to_datetime` outer-join strategy across native sensor clocks (4 Hz EDA, 4 Hz TEMP, 700 Hz label).

4. **Evaluation Paradigms:**
   - **Validation Scheme A (Naive Random Split):** 80/20 train/test split with ad-hoc standard scaling.
   - **Validation Scheme B (LOSO Cross-Validation):** 15-fold Leave-One-Subject-Out cross-validation with leakage-free in-fold scaling.
   - **Multi-Tier Comparative Analysis:** Comparative evaluation against Full Modality (155 features), Sliced Modality (62 features), and EDA+ACC (56 features).

In [1]:
# ==============================================================================
# 1. SYSTEM IMPORTS & RESOURCE MANAGEMENT
# ==============================================================================
import gc
import pickle
import sys
import time
import warnings
from pathlib import Path
from unittest.mock import MagicMock

# ==============================================================================
# 2. NUMERICAL, SIGNAL PROCESSING & PHYSIOLOGY LIBRARIES
# ==============================================================================
import matplotlib
matplotlib.use('Agg')
import numpy as np
import pandas as pd
from scipy.signal import resample_poly
import neurokit2 as nk

# Suppress harmless runtime warnings
warnings.simplefilter('ignore')

# Mock tkinter for biosppy headless execution
sys.modules['biosppy.inter_plotting'] = MagicMock()
sys.modules['biosppy.inter_plotting.acc'] = MagicMock()
import biosppy

# ==============================================================================
# 3. REPOSITORY & EXTERNAL HELPER CODE PATH DISCOVERY
# ==============================================================================
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'WESAD').exists())

FB_CODE_CANDIDATES = [
    REPO_ROOT / 'Scripts' / '3rd try' / 'fb_code',
    REPO_ROOT / '_forked' / 'fb_code',
    Path.cwd() / 'fb_code',
]
FB_CODE_DIR = next(
    path for path in FB_CODE_CANDIDATES
    if (path / 'respiration.py').exists() and (path / 'utils.py').exists()
)
if str(FB_CODE_DIR) not in sys.path:
    sys.path.insert(0, str(FB_CODE_DIR))

from utils import (
    butter_lowpass,
    butter_lowpass_filter,
    filterSignalFIR as filter_signal_fir,
    get_absolute_integral,
    get_dynamic_range,
    get_net_accel,
    get_peak_freq,
    get_slope,
    get_window_stats,
)

# ==============================================================================
# 4. SAMPLING FREQUENCIES, WINDOWING PARAMETERS & DIRECTORY SETUP
# ==============================================================================
# Sliced Modalities: Only Wrist ACC (32 Hz), Wrist EDA (4 Hz), Wrist TEMP (4 Hz), and Label (700 Hz)
RAW_FS = {'EDA': 4, 'TEMP': 4, 'label': 700}
FS = {'label': 700}

WINDOW_LENGTH_SECONDS = 60
WINDOW_SHIFT_SECONDS = 6
ACC_WINDOW_LENGTH_SECONDS = 5.0
ACC_WINDOW_SHIFT_SECONDS = 0.25

# Binary anxiety classification boundaries:
#   Class 1 (Anxiety Increased):   delta > 0
#   Class 0 (No Anxiety Increase): delta <= 0
LABEL_MAP = {
    0: 'No Anxiety Increase',
    1: 'Anxiety Increased'
}
LABEL_NAMES = [LABEL_MAP[0], LABEL_MAP[1]]

WESAD_DIR = REPO_ROOT / 'WESAD'
OUTPUT_DIR = REPO_ROOT / 'Data' / '6th try' / '2class' / 'Minimum_sensor_modals/eda_temp'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR = REPO_ROOT / 'Models' / '6th try' / '2class' / 'Minimum_sensor_modals/eda_temp'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

SUBJECTS = [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17]
PREVIEW_SUBJECT = 10
SKIP_EXTRACTED_SUBJECTS = True
OVERWRITE_EXISTING_OUTPUTS = False

print('Setup completed (Minimum Sensor Modalities: Wrist EDA and Wrist TEMP only):')
print(f'  Repository Root: {REPO_ROOT}')
print(f'  WESAD Directory: {WESAD_DIR}')
print(f'  Data Output Directory: {OUTPUT_DIR}')
print(f'  Model Output Directory: {MODELS_DIR}')


Setup completed (Minimum Sensor Modalities: Wrist EDA and Wrist TEMP only):
  Repository Root: E:\Rizqy\Kuliah\Skripsi\repo-skripsi
  WESAD Directory: E:\Rizqy\Kuliah\Skripsi\repo-skripsi\WESAD
  Data Output Directory: E:\Rizqy\Kuliah\Skripsi\repo-skripsi\Data\6th try\2class\Minimum_sensor_modals\eda_temp
  Model Output Directory: E:\Rizqy\Kuliah\Skripsi\repo-skripsi\Models\6th try\2class\Minimum_sensor_modals\eda_temp


## 1. Questionnaire & STAI Parser

Parses the WESAD shortened 6-item STAI questionnaire files (`S{id}_quest.csv`), performs required reverse-scoring on positively worded items (items 1, 4, 6), and computes condition score sums and score shifts ($\Delta = \text{condition\_stai\_sum} - \text{baseline\_stai\_sum}$).

In [2]:
# ==============================================================================
# 1. TIME NOTATION PARSER
# ==============================================================================
def parse_clock_mmss(value):
    text = str(value).strip()
    if not text:
        return np.nan
    if '.' in text:
        minute_text, second_text = text.split('.', 1)
        minutes = int(minute_text)
        seconds = int(second_text)
    else:
        minutes, seconds = int(text), 0
    if not 0 <= seconds < 60:
        raise ValueError(f'Invalid MM.SS time {value!r}: seconds must be 00-59')
    return minutes * 60 + seconds

# ==============================================================================
# 2. NUMERIC FIELD ALIGNMENT HELPER
# ==============================================================================
def _parse_numeric_fields(parts):
    values = []
    for part in parts:
        text = part.strip()
        if not text:
            values.append(np.nan)
        else:
            try:
                values.append(float(text))
            except ValueError:
                values.append(np.nan)
    return values

# ==============================================================================
# 3. STAI QUESTIONNAIRE DEFINITIONS & REVERSE-SCORING LOGIC
# ==============================================================================
STAI_ITEM_NAMES = (
    'feel_at_ease', 'feel_nervous', 'jittery', 'relaxed', 'worried', 'pleasant'
)
STAI_REVERSE_ITEM_INDICES = {0, 3, 5}

def score_stai_responses(raw_responses):
    raw = [float(value) if pd.notna(value) else np.nan for value in raw_responses]
    if len(raw) > len(STAI_ITEM_NAMES):
        raise ValueError(f'Expected at most {len(STAI_ITEM_NAMES)} shortened STAI items, got {len(raw)}')
    scored = [
        (5.0 - value) if index in STAI_REVERSE_ITEM_INDICES and pd.notna(value) else value
        for index, value in enumerate(raw)
    ]
    return raw, scored

# ==============================================================================
# 4. PARSER FOR PARTICIPANT QUESTIONNAIRE FILES (S{id}_quest.csv)
# ==============================================================================
def parse_quest_file(subject_id):
    quest_path = WESAD_DIR / f'S{subject_id}' / f'S{subject_id}_quest.csv'
    if not quest_path.exists():
        raise FileNotFoundError(f'Questionnaire file not found: {quest_path}')

    order = start_raw = end_raw = None
    stai_rows = []
    with open(quest_path, 'r', encoding='latin1') as f:
        for line in f:
            parts = [part.strip() for part in line.rstrip('\r\n').split(';')]
            tag = parts[0].strip() if parts else ''
            if tag.startswith('# ORDER'):
                order = [value.strip() for value in parts[1:]]
            elif tag.startswith('# START'):
                start_raw = parts[1:]
            elif tag.startswith('# END'):
                end_raw = parts[1:]
            elif tag.startswith('# STAI'):
                stai_rows.append(_parse_numeric_fields(parts[1:]))

    if order is None or start_raw is None or end_raw is None:
        raise ValueError(f'{quest_path.name}: missing ORDER, START, or END row')
    if len(start_raw) < len(order) or len(end_raw) < len(order):
        raise ValueError(f'{quest_path.name}: timing rows do not cover all conditions')
    if not stai_rows:
        raise ValueError(f'{quest_path.name}: no STAI response rows found')

    conditions = []
    for index, condition in enumerate(order):
        if not condition:
            continue
        start_text, end_text = start_raw[index].strip(), end_raw[index].strip()
        start_seconds = parse_clock_mmss(start_text) if start_text else np.nan
        end_seconds = parse_clock_mmss(end_text) if end_text else np.nan
        responses = [row[index] if index < len(row) else np.nan for row in stai_rows]
        raw_responses, scored_responses = score_stai_responses(responses)
        valid_raw = [float(value) for value in raw_responses if pd.notna(value)]
        valid_scored = [float(value) for value in scored_responses if pd.notna(value)]
        if any(value < 1 or value > 4 for value in valid_raw):
            raise ValueError(f'{quest_path.name}: STAI response outside 1-4 for condition {condition!r}')
        conditions.append({
            'condition': condition,
            'start_seconds': start_seconds,
            'end_seconds': end_seconds,
            'stai_responses_raw': valid_raw,
            'stai_responses_scored': valid_scored,
            'stai_item_count': len(valid_scored),
            'stai_mean_raw': float(np.mean(valid_raw)) if valid_raw else np.nan,
            'stai_mean': float(np.mean(valid_scored)) if valid_scored else np.nan,
            'stai_sum': float(np.sum(valid_scored)) if valid_scored else np.nan,
        })

    if not any(item['condition'].strip().lower() == 'base' for item in conditions):
        raise ValueError(f'{quest_path.name}: no Base condition found')
    return conditions

# ==============================================================================
# 5. BATCH PARSE & INSPECT QUESTIONNAIRE DATA ACROSS ALL SUBJECTS
# ==============================================================================
all_participant_conditions = []
for subject_id in SUBJECTS:
    participant_df = pd.DataFrame(parse_quest_file(subject_id))
    participant_df.insert(0, 'subject_id', subject_id)
    all_participant_conditions.append(participant_df)

all_conditions_df = pd.concat(all_participant_conditions, ignore_index=True)
display(all_conditions_df.sort_values(['subject_id', 'condition']).reset_index(drop=True))


,subject_id,condition,start_seconds,end_seconds,stai_responses_raw,stai_responses_scored,stai_item_count,stai_mean_raw,stai_mean,stai_sum
0,2,Base,428,1592,"[3.0, 1.0, 4.0, 3.0, 4.0]","[2.0, 1.0, 4.0, 2.0, 4.0]",5,3.0,2.6,13.0
1,2,Fun,4885,5267,"[4.0, 1.0, 4.0, 3.0, 4.0]","[1.0, 1.0, 4.0, 2.0, 4.0]",5,3.2,2.4,12.0
2,2,Medi 1,4219,4621,"[1.0, 2.0, 1.0, 1.0, 1.0]","[4.0, 2.0, 1.0, 4.0, 1.0]",5,1.2,2.4,12.0
3,2,Medi 2,5618,6015,"[2.0, 3.0, 2.0, 1.0, 1.0]","[3.0, 3.0, 2.0, 4.0, 1.0]",5,1.8,2.6,13.0
4,2,TSST,2395,3003,"[2.0, 3.0, 1.0, 1.0, 1.0]","[3.0, 3.0, 1.0, 4.0, 1.0]",5,1.6,2.4,12.0
...,...,...,...,...,...,...,...,...,...,...
114,17,Medi 2,5349,5763,"[1.0, 1.0, 1.0, 4.0, 1.0]","[4.0, 1.0, 1.0, 1.0, 1.0]",5,1.6,1.6,8.0
115,17,TSST,3602,4345,"[2.0, 4.0, 3.0, 1.0, 2.0]","[3.0, 4.0, 3.0, 4.0, 2.0]",5,2.4,3.2,16.0
116,17,bRead,1417,1495,"[2.0, 4.0, 2.0, 1.0, 1.0]","[3.0, 4.0, 2.0, 4.0, 1.0]",5,2.0,2.8,14.0
117,17,fRead,2282,2359,[],[],0,NaN,NaN,NaN


## 2. Signal Preprocessing & Minimum-Sensor STAI Labeling Pipeline (EDA + TEMP)

Implements the 2-class baseline-relative labeling logic and prepares synchronized signals strictly for wrist EDA and wrist TEMP.

In [3]:
# ==============================================================================
# 1. RAW SUBJECT PICKLE LOADER
# ==============================================================================
def load_raw_subject(subject_id):
    path = WESAD_DIR / f'S{subject_id}' / f'S{subject_id}.pkl'
    with open(path, 'rb') as file:
        return pickle.load(file, encoding='latin1')

# ==============================================================================
# 2. RAW SIGNAL EXTRACTION & PREPROCESSING (EDA + TEMP ONLY)
# ==============================================================================
def compute_features_stai(subject_id):
    raw_data = load_raw_subject(subject_id)
    wrist = raw_data['signal']['wrist']
    labels = np.asarray(raw_data['label']).ravel()

    eda_df = pd.DataFrame(wrist['EDA'].astype(np.float32), columns=['EDA'])
    eda_df['EDA'] = butter_lowpass_filter(eda_df['EDA'], 1.0, RAW_FS['EDA'], 6)
    eda_data = nk.eda_phasic(nk.standardize(eda_df['EDA']), sampling_rate=RAW_FS['EDA'])
    eda_df['EDA_SCR'] = eda_data['EDA_Phasic'].astype(np.float32)
    eda_df['EDA_SCL'] = eda_data['EDA_Tonic'].astype(np.float32)

    temp_df = pd.DataFrame(wrist['TEMP'].astype(np.float32), columns=['TEMP'])

    # Build nanosecond timestamp indices for EDA, TEMP, and Labels
    signals_with_freq = [
        (eda_df, RAW_FS['EDA']), (temp_df, RAW_FS['TEMP']),
        (pd.DataFrame({'protocol_label': labels.astype(np.int8)}), RAW_FS['label']),
    ]
    aligned_blocks = []
    for sig_df, freq in signals_with_freq:
        ts = pd.to_datetime(np.round(np.arange(len(sig_df), dtype=np.float64) * (1e9 / freq)).astype(np.int64), unit='ns')
        aligned_blocks.append(sig_df.set_axis(ts))

    df = pd.concat(aligned_blocks, axis=1)
    df['protocol_label'] = df['protocol_label'].bfill()
    df.dropna(subset=['protocol_label'], inplace=True)
    return df

print('Retained modalities: Wrist EDA (4 Hz) and Wrist TEMP (4 Hz) only')


Retained modalities: Wrist EDA (4 Hz) and Wrist TEMP (4 Hz) only


## 3. Sliding Window Minimum Feature Extraction & Saving Pickles (EDA + TEMP)

Extracts 60-second sliding windows (shifted by 6 seconds) and computes **20 predictive features** (14 wrist EDA + 6 wrist TEMP).

In [4]:
def get_window_stats(data, label=-1):
    return {
        'mean': data.mean(),
        'std': data.std(),
        'min': data.min(),
        'max': data.max(),
        'label': label
    }

# ==============================================================================
# 1. SUBJECT-LEVEL SLIDING WINDOW FEATURE EXTRACTION PIPELINE (EDA + TEMP ONLY)
# ==============================================================================
def extract_stai_features_subject(subject_id):
    # Check if full dataset exists to accelerate slicing, otherwise compute from raw signals
    full_path = REPO_ROOT / 'Data' / '6th try' / '2class' / f'S{subject_id}_stai_2class_60s_shift6s.pkl'
    if full_path.exists():
        df_full = pd.read_pickle(full_path)
        wrist_eda = [c for c in df_full.columns if 'EDA' in c and '_C' not in c]
        wrist_temp = [c for c in df_full.columns if 'TEMP' in c and '_C' not in c]
        meta = [
            'anxiety_label', 'delta_stai', 'subject_id', 'condition',
            'window_start_seconds', 'window_end_seconds', 'condition_stai_mean',
            'baseline_stai_mean', 'delta_stai_sum', 'delta_stai_mean',
            'condition_stai_sum', 'baseline_stai_sum'
        ]
        min_cols = wrist_eda + wrist_temp + [c for c in meta if c in df_full.columns]
        result = df_full[min_cols].copy()
        result.attrs = getattr(df_full, 'attrs', {}).copy()
        return result

    df = compute_features_stai(subject_id)
    condition_labels = build_condition_labels(subject_id)
    samples = []
    total_duration_sec = (df.index[-1] - df.index[0]).total_seconds()
    candidate_windows = max(0, int((total_duration_sec - WINDOW_LENGTH_SECONDS) // WINDOW_SHIFT_SECONDS + 1))
    dropped_unlabeled_windows = 0
    dropped_ambiguous_windows = 0
    feat_names = None

    for step_idx in range(candidate_windows):
        start_sec = step_idx * WINDOW_SHIFT_SECONDS
        end_sec = start_sec + WINDOW_LENGTH_SECONDS
        containing = [
            item for item in condition_labels
            if start_sec >= item['start_seconds'] and end_sec <= item['end_seconds']
        ]
        if len(containing) != 1:
            if len(containing) > 1:
                dropped_ambiguous_windows += 1
            else:
                dropped_unlabeled_windows += 1
            continue
        condition = containing[0]
        ts_start = pd.to_datetime(int(start_sec * 1e9), unit='ns')
        ts_end = pd.to_datetime(int(end_sec * 1e9), unit='ns')
        window = df.loc[ts_start:ts_end].drop(columns=['protocol_label']).copy()

        stats = get_window_stats(window[['EDA', 'EDA_SCR', 'EDA_SCL', 'TEMP']], label=0)
        if 'label' in stats:
            del stats['label']
        x = pd.DataFrame(stats)
        if feat_names is None:
            feat_names = [f'{row}_{col}' for row in x.index for col in x.columns]
        wdf = pd.DataFrame(x.values.flatten()).T
        wdf.columns = feat_names

        wdf['anxiety_label'] = condition['anxiety_label']
        wdf['delta_stai'] = condition['delta_stai']
        wdf['subject_id'] = subject_id
        wdf['condition'] = condition['condition']
        wdf['window_start_seconds'] = start_sec
        wdf['window_end_seconds'] = end_sec
        wdf['condition_stai_mean'] = condition['stai_mean']
        wdf['baseline_stai_mean'] = condition['baseline_stai_mean']
        wdf['delta_stai_sum'] = condition['delta_stai_sum']
        wdf['delta_stai_mean'] = condition['delta_stai_mean']
        wdf['condition_stai_sum'] = condition['condition_stai_sum']
        wdf['baseline_stai_sum'] = condition['baseline_stai_sum']

        wdf['EDA_slope'] = get_slope(window['EDA'].dropna())
        wdf['EDA_drange'] = get_dynamic_range(window['EDA'].dropna())
        wdf['TEMP_slope'] = get_slope(window['TEMP'].dropna())
        wdf['TEMP_drange'] = get_dynamic_range(window['TEMP'].dropna())

        samples.append(wdf)

    result = pd.concat(samples, ignore_index=True) if samples else pd.DataFrame()
    result.attrs['candidate_windows'] = candidate_windows
    result.attrs['dropped_unlabeled_windows'] = dropped_unlabeled_windows
    result.attrs['dropped_ambiguous_windows'] = dropped_ambiguous_windows
    return result

# ==============================================================================
# 2. BATCH PROCESSING & SAVING OF MINIMUM SENSOR (EDA + TEMP) DATASETS
# ==============================================================================
summary = []
for subject_id in SUBJECTS:
    out_path = OUTPUT_DIR / f'S{subject_id}_stai_2class_60s_shift6s_minimum_eda_temp.pkl'
    if out_path.exists() and SKIP_EXTRACTED_SUBJECTS:
        df_existing = pd.read_pickle(out_path)
        summary.append({
            'subject': subject_id,
            'candidate_windows': len(df_existing),
            'retained_windows': len(df_existing),
            'dropped_unlabeled_windows': 0,
            'dropped_ambiguous_windows': 0,
            'n_features': len(df_existing.columns),
            'no_anxiety_windows (0)': int((df_existing['anxiety_label'] == 0).sum()),
            'anxiety_windows (1)': int((df_existing['anxiety_label'] == 1).sum()),
            'saved_to': out_path.name,
            'seconds': 0.0,
        })
        print(f'S{subject_id}: existing minimum eda_temp dataset loaded ({len(df_existing)} windows).')
        continue
    if out_path.exists() and not OVERWRITE_EXISTING_OUTPUTS:
        print(f'WARNING: {out_path} exists and overwrite is disabled; skipping S{subject_id}.')
        continue

    start_t = time.time()
    features_df = extract_stai_features_subject(subject_id)
    if features_df.empty:
        print(f'S{subject_id}: no fully-contained labeled windows; no file saved')
        continue
    features_df.to_pickle(out_path)
    summary.append({
        'subject': subject_id,
        'candidate_windows': features_df.attrs.get('candidate_windows', len(features_df)),
        'retained_windows': len(features_df),
        'dropped_unlabeled_windows': features_df.attrs.get('dropped_unlabeled_windows', 0),
        'dropped_ambiguous_windows': features_df.attrs.get('dropped_ambiguous_windows', 0),
        'n_features': len(features_df.columns),
        'no_anxiety_windows (0)': int((features_df['anxiety_label'] == 0).sum()),
        'anxiety_windows (1)': int((features_df['anxiety_label'] == 1).sum()),
        'saved_to': out_path.name,
        'seconds': round(time.time() - start_t, 2),
    })
    print(summary[-1])
    del features_df
    gc.collect()

pd.DataFrame(summary)


S2: existing minimum eda_temp dataset loaded (257 windows).
S3: existing minimum eda_temp dataset loaded (260 windows).
S4: existing minimum eda_temp dataset loaded (272 windows).


S5: existing minimum eda_temp dataset loaded (270 windows).
S6: existing minimum eda_temp dataset loaded (269 windows).
S7: existing minimum eda_temp dataset loaded (275 windows).
S8: existing minimum eda_temp dataset loaded (263 windows).


S9: existing minimum eda_temp dataset loaded (262 windows).
S10: existing minimum eda_temp dataset loaded (289 windows).


S11: existing minimum eda_temp dataset loaded (285 windows).
S13: existing minimum eda_temp dataset loaded (280 windows).
S14: existing minimum eda_temp dataset loaded (282 windows).
S15: existing minimum eda_temp dataset loaded (282 windows).
S16: existing minimum eda_temp dataset loaded (274 windows).
S17: existing minimum eda_temp dataset loaded (275 windows).


,subject,candidate_windows,retained_windows,dropped_unlabeled_windows,dropped_ambiguous_windows,n_features,no_anxiety_windows (0),anxiety_windows (1),saved_to,seconds
0,2,257,257,0,0,32,166,91,S2_stai_2class_60s_shift6s_minimum_eda_temp.pkl,0.0
1,3,260,260,0,0,32,48,212,S3_stai_2class_60s_shift6s_minimum_eda_temp.pkl,0.0
2,4,272,272,0,0,32,55,217,S4_stai_2class_60s_shift6s_minimum_eda_temp.pkl,0.0
3,5,270,270,0,0,32,113,157,S5_stai_2class_60s_shift6s_minimum_eda_temp.pkl,0.0
4,6,269,269,0,0,32,111,158,S6_stai_2class_60s_shift6s_minimum_eda_temp.pkl,0.0
5,7,275,275,0,0,32,56,219,S7_stai_2class_60s_shift6s_minimum_eda_temp.pkl,0.0
6,8,263,263,0,0,32,103,160,S8_stai_2class_60s_shift6s_minimum_eda_temp.pkl,0.0
7,9,262,262,0,0,32,113,149,S9_stai_2class_60s_shift6s_minimum_eda_temp.pkl,0.0
8,10,289,289,0,0,32,110,179,S10_stai_2class_60s_shift6s_minimum_eda_temp.pkl,0.0
9,11,285,285,0,0,32,178,107,S11_stai_2class_60s_shift6s_minimum_eda_temp.pkl,0.0


## 4. Preview a Minimum-Sensor (EDA + TEMP) STAI-Labeled Subject Dataset

Inspects a single subject dataset (S10) to verify feature dimensions (20 predictive features + 12 metadata columns = 32 total columns).

In [5]:
preview_path = OUTPUT_DIR / f'S{PREVIEW_SUBJECT}_stai_2class_60s_shift6s_minimum_eda_temp.pkl'
preview_df = pd.read_pickle(preview_path)
print(f'=== MINIMUM-SENSOR (EDA + TEMP) STAI-LABELED SUBJECT DATASET PREVIEW (S{PREVIEW_SUBJECT}) ===')
print(f'Shape: {preview_df.shape} (Windows: {preview_df.shape[0]}, Total Columns: {preview_df.shape[1]})')
display(preview_df.head())


=== MINIMUM-SENSOR (EDA + TEMP) STAI-LABELED SUBJECT DATASET PREVIEW (S10) ===
Shape: (289, 32) (Windows: 289, Total Columns: 32)


,EDA_mean,EDA_std,EDA_min,EDA_max,EDA_SCR_mean,EDA_SCR_std,EDA_SCR_min,EDA_SCR_max,EDA_SCL_mean,EDA_SCL_std,...,subject_id,condition,window_start_seconds,window_end_seconds,condition_stai_mean,baseline_stai_mean,delta_stai_sum,delta_stai_mean,condition_stai_sum,baseline_stai_sum
0,0.431412,0.001997,0.425672,0.436552,0.000146,0.002141,-0.006167,0.008121,-0.717431,0.001415,...,10,Fun,1674,1734,1.3333,1.8333,-3.0,-0.5,8.0,11.0
1,0.431220,0.001893,0.425819,0.436552,-0.000009,0.001731,-0.006167,0.003883,-0.717526,0.001549,...,10,Fun,1680,1740,1.3333,1.8333,-3.0,-0.5,8.0,11.0
2,0.431046,0.002073,0.425460,0.436552,0.000012,0.001776,-0.006167,0.003883,-0.717774,0.001868,...,10,Fun,1686,1746,1.3333,1.8333,-3.0,-0.5,8.0,11.0
3,0.430695,0.002150,0.425460,0.436552,-0.000003,0.001733,-0.006167,0.003883,-0.718217,0.002006,...,10,Fun,1692,1752,1.3333,1.8333,-3.0,-0.5,8.0,11.0
4,0.430187,0.001848,0.425292,0.434346,-0.000033,0.001762,-0.006167,0.003883,-0.718849,0.001705,...,10,Fun,1698,1758,1.3333,1.8333,-3.0,-0.5,8.0,11.0


## 5. Verify Modality Breakdown (EDA + TEMP Only)

Verifies that only wrist EDA (14 features) and wrist TEMP (6 features) are present, with 0 ACC, 0 chest, and 0 BVP features.

In [6]:
metadata_columns = {
    'anxiety_label', 'binary_anxiety', 'three_class_anxiety', 'subject_id',
    'condition', 'window_start_seconds', 'window_end_seconds',
    'condition_stai_mean', 'baseline_stai_mean', 'delta_stai',
    'condition_stai_sum', 'baseline_stai_sum', 'delta_stai_sum', 'delta_stai_mean',
}
pred_features = [col for col in preview_df.columns if col not in metadata_columns]

acc_features = [col for col in pred_features if any(k in col for k in ['ACC', 'net_acc'])]
eda_features = [col for col in pred_features if 'EDA' in col]
temp_features = [col for col in pred_features if 'TEMP' in col]
chest_features = [col for col in pred_features if '_C' in col or any(k in col for k in ['ECG', 'EMG', 'Resp'])]
bvp_features = [col for col in pred_features if 'BVP' in col]

print(f'Total Predictive Minimum Features: {len(pred_features)}')
print(f'  - Wrist Electrodermal Activity (EDA) features: {len(eda_features)}')
print(f'  - Wrist Skin Temperature (TEMP) features: {len(temp_features)}')
print(f'  - Wrist Accelerometer (ACC) features: {len(acc_features)}')
print(f'  - Chest sensors (ECG/EMG/Resp/Chest EDA/Temp/ACC): {len(chest_features)}')
print(f'  - BVP sensors: {len(bvp_features)}')

assert len(pred_features) == 20, f'Expected 20 predictive features, got {len(pred_features)}'
assert len(acc_features) == 0, 'ACC features must be 0 in EDA+TEMP setup!'
assert len(chest_features) == 0, 'Chest features must be 0 in minimum sensor setup!'
assert len(bvp_features) == 0, 'BVP features must be 0 in minimum sensor setup!'
print('\nAssertion verified: Dataset strictly contains only wrist EDA (14) and wrist TEMP (6) features.')


Total Predictive Minimum Features: 20
  - Wrist Electrodermal Activity (EDA) features: 14
  - Wrist Skin Temperature (TEMP) features: 6
  - Wrist Accelerometer (ACC) features: 0
  - Chest sensors (ECG/EMG/Resp/Chest EDA/Temp/ACC): 0
  - BVP sensors: 0

Assertion verified: Dataset strictly contains only wrist EDA (14) and wrist TEMP (6) features.


## 6. Minimum-Sensor (EDA + TEMP) Model Training & Evaluation (Validation Scheme A: Naive Random Split 80/20)

In [7]:
# ==============================================================================
# 1. MACHINE LEARNING & EVALUATION IMPORTS
# ==============================================================================
import joblib
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

# ==============================================================================
# 2. MODEL CONFIGURATION & HYPERPARAMETERS
# ==============================================================================
TEST_SIZE = 0.2          # 20% random split matching Alkurdi
RANDOM_STATE = 0         # random_state=0 matching Alkurdi
KNN_N_NEIGHBORS = 5
RF_MAX_DEPTH = 25

# ==============================================================================
# 3. DATASET LOADING & COMBINATION
# ==============================================================================
pkl_files = sorted(OUTPUT_DIR.glob('S*_stai_2class_60s_shift6s_minimum_eda_temp.pkl'))
if not pkl_files:
    raise FileNotFoundError(f'No S*_stai_2class_60s_shift6s_minimum_eda_temp.pkl datasets found in {OUTPUT_DIR}. Run Section 3 first.')

all_dfs = []
for path in pkl_files:
    subject_df = pd.read_pickle(path)
    all_dfs.append(subject_df)

combined_df = pd.concat(all_dfs, ignore_index=True)
combined_df = combined_df[combined_df['anxiety_label'].isin([0, 1])].copy()

# ==============================================================================
# 4. FEATURE / TARGET SEPARATION & DATA CLEANING
# ==============================================================================
metadata_columns = {
    'anxiety_label', 'binary_anxiety', 'three_class_anxiety', 'subject_id',
    'condition', 'window_start_seconds', 'window_end_seconds',
    'condition_stai_mean', 'baseline_stai_mean', 'delta_stai',
    'condition_stai_sum', 'baseline_stai_sum', 'delta_stai_sum', 'delta_stai_mean',
}
feature_columns = [col for col in combined_df.columns if col not in metadata_columns]
X = combined_df[feature_columns].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)
y = combined_df['anxiety_label'].astype(int)
groups = combined_df['subject_id'].astype(int)

class_counts = y.value_counts().sort_index()
print(f'Loaded {len(pkl_files)} sliced STAI-labeled subjects; combined shape={combined_df.shape}')
print('Class distribution (0=No Anxiety, 1=Anxiety Increased):')
display(class_counts.rename(index=LABEL_MAP).rename_axis('anxiety_label').to_frame('windows'))

# ==============================================================================
# 5. VALIDATION SCHEME A: NAIVE RANDOM SPLIT (DATA LEAKAGE REPLICATION)
# ==============================================================================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)
print(f'Validation Scheme A (Naive Random Split) - Train: {X_train.shape}, Test: {X_test.shape}')

sc = StandardScaler()
X_train_scaled = sc.fit_transform(X_train)
X_test_scaled = sc.transform(X_test)

models = {
    'LDA': LinearDiscriminantAnalysis(),
    'Random Forest': RandomForestClassifier(max_depth=RF_MAX_DEPTH, random_state=RANDOM_STATE),
    'SVM Linear': SVC(kernel='linear', C=1.0, random_state=RANDOM_STATE),
    'SVM': SVC(kernel='rbf', random_state=RANDOM_STATE),
    'XGBoost': XGBClassifier(n_estimators=100, eval_metric='logloss', random_state=RANDOM_STATE),
    'KNN': KNeighborsClassifier(n_neighbors=KNN_N_NEIGHBORS),
}

scheme_a_results = {}
for name, model in models.items():
    print(f'\n--- Training {name} ---')
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    cr = classification_report(y_test, y_pred, labels=[0, 1], target_names=LABEL_NAMES, zero_division=0)
    scheme_a_results[name] = {'accuracy': acc, 'confusion_matrix': cm, 'report': cr}
    print(f'{name} Accuracy: {acc:.4f}')
    print('Confusion Matrix (rows=true, cols=pred; [0, 1]):\n', cm)
    print('Classification Report:\n', cr)

    safe_name = name.lower().replace(' ', '_').replace('(', '').replace(')', '').replace('=', '_').replace(':', '')
    model_filename = MODELS_DIR / f'stai_2class_schemeA_minimum_eda_temp_{safe_name}_model.joblib'
    joblib.dump({'model': model, 'scaler': sc, 'features': feature_columns}, model_filename)
    print(f'Saved model to {model_filename.name}')

print('\nValidation Scheme A evaluation completed.')


Loaded 15 sliced STAI-labeled subjects; combined shape=(4095, 32)
Class distribution (0=No Anxiety, 1=Anxiety Increased):


,windows
anxiety_label,
No Anxiety Increase,1785
Anxiety Increased,2310


Validation Scheme A (Naive Random Split) - Train: (3276, 20), Test: (819, 20)

--- Training LDA ---
LDA Accuracy: 0.7766
Confusion Matrix (rows=true, cols=pred; [0, 1]):
 [[333  26]
 [157 303]]
Classification Report:
                      precision    recall  f1-score   support

No Anxiety Increase       0.68      0.93      0.78       359
  Anxiety Increased       0.92      0.66      0.77       460

           accuracy                           0.78       819
          macro avg       0.80      0.79      0.78       819
       weighted avg       0.82      0.78      0.78       819

Saved model to stai_2class_schemeA_minimum_eda_temp_lda_model.joblib

--- Training Random Forest ---


Random Forest Accuracy: 0.9963
Confusion Matrix (rows=true, cols=pred; [0, 1]):
 [[356   3]
 [  0 460]]
Classification Report:
                      precision    recall  f1-score   support

No Anxiety Increase       1.00      0.99      1.00       359
  Anxiety Increased       0.99      1.00      1.00       460

           accuracy                           1.00       819
          macro avg       1.00      1.00      1.00       819
       weighted avg       1.00      1.00      1.00       819

Saved model to stai_2class_schemeA_minimum_eda_temp_random_forest_model.joblib

--- Training SVM Linear ---


SVM Linear Accuracy: 0.7851
Confusion Matrix (rows=true, cols=pred; [0, 1]):
 [[323  36]
 [140 320]]
Classification Report:
                      precision    recall  f1-score   support

No Anxiety Increase       0.70      0.90      0.79       359
  Anxiety Increased       0.90      0.70      0.78       460

           accuracy                           0.79       819
          macro avg       0.80      0.80      0.79       819
       weighted avg       0.81      0.79      0.79       819

Saved model to stai_2class_schemeA_minimum_eda_temp_svm_linear_model.joblib

--- Training SVM ---


SVM Accuracy: 0.8596
Confusion Matrix (rows=true, cols=pred; [0, 1]):
 [[343  16]
 [ 99 361]]
Classification Report:
                      precision    recall  f1-score   support

No Anxiety Increase       0.78      0.96      0.86       359
  Anxiety Increased       0.96      0.78      0.86       460

           accuracy                           0.86       819
          macro avg       0.87      0.87      0.86       819
       weighted avg       0.88      0.86      0.86       819

Saved model to stai_2class_schemeA_minimum_eda_temp_svm_model.joblib

--- Training XGBoost ---


XGBoost Accuracy: 0.9976
Confusion Matrix (rows=true, cols=pred; [0, 1]):
 [[357   2]
 [  0 460]]
Classification Report:
                      precision    recall  f1-score   support

No Anxiety Increase       1.00      0.99      1.00       359
  Anxiety Increased       1.00      1.00      1.00       460

           accuracy                           1.00       819
          macro avg       1.00      1.00      1.00       819
       weighted avg       1.00      1.00      1.00       819

Saved model to stai_2class_schemeA_minimum_eda_temp_xgboost_model.joblib

--- Training KNN ---
KNN Accuracy: 0.9756
Confusion Matrix (rows=true, cols=pred; [0, 1]):
 [[348  11]
 [  9 451]]
Classification Report:
                      precision    recall  f1-score   support

No Anxiety Increase       0.97      0.97      0.97       359
  Anxiety Increased       0.98      0.98      0.98       460

           accuracy                           0.98       819
          macro avg       0.98      0.97      0.98

## 7. Subject-Independent Evaluation (Validation Scheme B: LOSO on EDA + TEMP)

Evaluates true generalized subject-independent performance under 15-fold Leave-One-Subject-Out (LOSO) cross-validation on the 20-feature EDA+TEMP set.

In [8]:
# ==============================================================================
# 1. LEAVE-ONE-SUBJECT-OUT (LOSO) VALIDATION SETUP (SCHEME B)
# ==============================================================================
import json
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score, classification_report
from sklearn.model_selection import LeaveOneGroupOut

loso_indices = np.arange(len(combined_df))
loso_y = combined_df['anxiety_label'].astype(int).to_numpy()
loso_groups = combined_df['subject_id'].astype(int).to_numpy()
loso_splitter = LeaveOneGroupOut()

loso_model_factories = {
    'Random Forest (max_depth=4)': lambda: RandomForestClassifier(max_depth=4, random_state=RANDOM_STATE),
    'Random Forest (max_depth=25)': lambda: RandomForestClassifier(max_depth=RF_MAX_DEPTH, random_state=RANDOM_STATE),
    'XGBoost': lambda: XGBClassifier(n_estimators=100, eval_metric='logloss', random_state=RANDOM_STATE),
    'SVM (RBF)': lambda: SVC(kernel='rbf', random_state=RANDOM_STATE),
    'KNN (k=5)': lambda: KNeighborsClassifier(n_neighbors=KNN_N_NEIGHBORS),
    'SVM Linear (C=1.0)': lambda: SVC(kernel='linear', C=1.0, random_state=RANDOM_STATE),
    'SVM Linear (C=0.9)': lambda: SVC(kernel='linear', C=0.9, random_state=RANDOM_STATE),
    'LDA': lambda: LinearDiscriminantAnalysis(),
}

loso_predictions = {name: np.full(len(combined_df), -1, dtype=int) for name in loso_model_factories}
loso_subjects = []

# ==============================================================================
# 2. LOSO CROSS-VALIDATION FOLD EXECUTION
# ==============================================================================
for train_idx, test_idx in loso_splitter.split(loso_indices, loso_y, loso_groups):
    held_out_subject = int(combined_df.iloc[test_idx]['subject_id'].iloc[0])
    loso_subjects.append(held_out_subject)
    fold_scaler = StandardScaler()
    X_train_fold = fold_scaler.fit_transform(X.iloc[train_idx])
    X_test_fold = fold_scaler.transform(X.iloc[test_idx])
    y_train_fold = loso_y[train_idx]
    for name, factory in loso_model_factories.items():
        model = factory()
        model.fit(X_train_fold, y_train_fold)
        loso_predictions[name][test_idx] = model.predict(X_test_fold)

# ==============================================================================
# 3. METRIC COMPUTATION & PER-SUBJECT DETAILED BREAKDOWN
# ==============================================================================
loso_summary = []
per_subject_loso_results = {}

for name, predictions in loso_predictions.items():
    pooled_acc = accuracy_score(loso_y, predictions)
    pooled_bal_acc = balanced_accuracy_score(loso_y, predictions)
    pooled_macro_f1 = f1_score(loso_y, predictions, average='macro', zero_division=0)
    per_class_f1 = f1_score(loso_y, predictions, average=None, labels=[0, 1], zero_division=0)

    per_subject_loso_results[name] = {}
    subj_accs, subj_bal_accs, subj_macro_f1s = [], [], []
    for s in loso_subjects:
        mask = (loso_groups == s)
        s_true = loso_y[mask]
        s_pred = predictions[mask]
        acc = float(accuracy_score(s_true, s_pred))
        macro_f1 = float(f1_score(s_true, s_pred, average='macro', zero_division=0))
        c_f1 = f1_score(s_true, s_pred, average=None, labels=[0, 1], zero_division=0)
        bal_acc = float(balanced_accuracy_score(s_true, s_pred)) if len(np.unique(s_true)) > 1 else None

        subj_accs.append(acc)
        subj_macro_f1s.append(macro_f1)
        if bal_acc is not None:
            subj_bal_accs.append(bal_acc)

        per_subject_loso_results[name][f"S{s}"] = {
            'windows': int(mask.sum()),
            'accuracy': round(acc, 4),
            'balanced_accuracy': round(bal_acc, 4) if bal_acc is not None else None,
            'f1_macro': round(macro_f1, 4),
            'f1_class0': round(float(c_f1[0]), 4),
            'f1_class1': round(float(c_f1[1]), 4),
        }

    loso_summary.append({
        'Model': name,
        'Pooled Accuracy': pooled_acc,
        'Pooled Balanced Acc': pooled_bal_acc,
        'Pooled Macro F1': pooled_macro_f1,
        'Class 0 F1 (No Anxiety)': per_class_f1[0],
        'Class 1 F1 (Anxiety Inc)': per_class_f1[1],
        'Macro-Subject Accuracy': float(np.mean(subj_accs)),
        'Macro-Subject Balanced Acc': float(np.mean(subj_bal_accs)) if subj_bal_accs else np.nan,
        'Macro-Subject Macro F1': float(np.mean(subj_macro_f1s)),
    })

loso_summary_df = pd.DataFrame(loso_summary)
print('LOSO Cross-Validation Summary (Validation Scheme B - Sliced Sensors: Wrist ACC, EDA, TEMP):')
display(loso_summary_df.round(4))

# ==============================================================================
# 4. CONSOLIDATED PER-SUBJECT BREAKDOWN ACROSS ARCHITECTURES & ARTIFACT EXPORT
# ==============================================================================
consolidated_loso_records = []
for s in loso_subjects:
    s_key = f"S{s}"
    n_win = per_subject_loso_results['Random Forest (max_depth=4)'][s_key]['windows']
    row_data = {'Subject': s_key, 'Total Windows': n_win}
    for m in ['Random Forest (max_depth=4)', 'Random Forest (max_depth=25)', 'XGBoost', 'SVM (RBF)', 'KNN (k=5)', 'SVM Linear (C=1.0)', 'LDA']:
        if m in per_subject_loso_results:
            acc = per_subject_loso_results[m][s_key]['accuracy']
            f1 = per_subject_loso_results[m][s_key]['f1_macro']
            row_data[f"{m} Acc"] = f"{acc:.4f}"
            row_data[f"{m} F1"] = f"{f1:.4f}"
    consolidated_loso_records.append(row_data)

consolidated_loso_df = pd.DataFrame(consolidated_loso_records).set_index('Subject')
print('\nLOSO Per-Subject Detailed Breakdown across Key Architectures:')
display(consolidated_loso_df)

# Export per-subject LOSO results to JSON
loso_results_export_path = MODELS_DIR / 'per_subject_loso_results.json'
with open(loso_results_export_path, 'w', encoding='utf-8') as f:
    json.dump(per_subject_loso_results, f, indent=2)
print(f'Saved per-subject LOSO results to: {loso_results_export_path}')

# Export evaluation summary
with open(MODELS_DIR / 'evaluation_summary.json', 'w', encoding='utf-8') as f:
    json.dump({
        'validation_scheme_a': {k: {'accuracy': round(v['accuracy'], 4)} for k, v in scheme_a_results.items()},
        'validation_scheme_b': loso_summary
    }, f, indent=2)


LOSO Cross-Validation Summary (Validation Scheme B - Sliced Sensors: Wrist ACC, EDA, TEMP):


,Model,Pooled Accuracy,Pooled Balanced Acc,Pooled Macro F1,Class 0 F1 (No Anxiety),Class 1 F1 (Anxiety Inc),Macro-Subject Accuracy,Macro-Subject Balanced Acc,Macro-Subject Macro F1
0,Random Forest (max_depth=4),0.6799,0.6812,0.6780,0.6533,0.7027,0.6809,0.7332,0.6676
1,Random Forest (max_depth=25),0.6950,0.6992,0.6940,0.6765,0.7115,0.6958,0.7321,0.6844
2,XGBoost,0.6686,0.6662,0.6650,0.6299,0.7000,0.6697,0.7004,0.6434
3,SVM (RBF),0.6921,0.6968,0.6912,0.6751,0.7074,0.6911,0.7169,0.6715
4,KNN (k=5),0.6662,0.6627,0.6619,0.6239,0.6999,0.6658,0.6645,0.6368
5,SVM Linear (C=1.0),0.6855,0.6922,0.6851,0.6736,0.6965,0.6840,0.7422,0.6685
6,SVM Linear (C=0.9),0.6860,0.6927,0.6855,0.6741,0.6970,0.6845,0.7426,0.6689
7,LDA,0.7118,0.7246,0.7118,0.7137,0.7099,0.7108,0.7653,0.7001



LOSO Per-Subject Detailed Breakdown across Key Architectures:


,Total Windows,Random Forest (max_depth=4) Acc,Random Forest (max_depth=4) F1,Random Forest (max_depth=25) Acc,Random Forest (max_depth=25) F1,XGBoost Acc,XGBoost F1,SVM (RBF) Acc,SVM (RBF) F1,KNN (k=5) Acc,KNN (k=5) F1,SVM Linear (C=1.0) Acc,SVM Linear (C=1.0) F1,LDA Acc,LDA F1
Subject,,,,,,,,,,,,,,,
S2,257,0.9377,0.9343,0.9611,0.9584,0.9066,0.9028,0.7121,0.7121,0.9222,0.9184,0.6887,0.6886,0.8288,0.8263
S3,260,0.4846,0.4777,0.5731,0.5546,0.4962,0.4879,0.4923,0.4845,0.4692,0.3535,0.3692,0.3692,0.4385,0.4358
S4,272,0.5772,0.5642,0.5515,0.5316,0.5772,0.5642,0.5772,0.5642,0.6544,0.6299,0.5772,0.5642,0.5772,0.5642
S5,270,0.6333,0.6284,0.6370,0.6325,0.7370,0.7369,0.6963,0.6952,0.7556,0.7556,0.5926,0.5555,0.7000,0.6889
S6,269,0.6022,0.5923,0.6022,0.5923,0.6283,0.6155,0.6171,0.6056,0.6208,0.6099,0.6394,0.6336,0.6320,0.6260
S7,275,0.5636,0.5468,0.5309,0.5121,0.4327,0.3020,0.5164,0.3405,0.4618,0.3159,0.5855,0.5719,0.5709,0.5591
S8,263,0.7719,0.7718,0.7719,0.7718,0.7529,0.7528,0.7719,0.7718,0.7719,0.7650,0.7719,0.7718,0.7719,0.7718
S9,262,0.7863,0.7850,0.7863,0.7850,0.7824,0.7813,0.8053,0.8047,0.5344,0.4208,0.7863,0.7850,0.7863,0.7850
S10,289,0.8270,0.8262,0.8270,0.8262,0.8028,0.8024,0.8374,0.8364,0.9170,0.9149,0.8270,0.8262,0.8270,0.8262


Saved per-subject LOSO results to: E:\Rizqy\Kuliah\Skripsi\repo-skripsi\Models\6th try\2class\Minimum_sensor_modals\eda_temp\per_subject_loso_results.json


## 8. Multi-Tier Shortfall Analysis & Modality Ablation (EDA + TEMP)

Compares EDA + TEMP (20 features) performance against:
- Full Modality Suite (155 features)
- Sliced Smartwatch Suite: ACC + EDA + TEMP (62 features)
- Minimum Sensor Suite: ACC + EDA (56 features)

In [9]:
# ==============================================================================
# 1. MULTI-TIER ACCURACY SHORTFALL & PERFORMANCE RETENTION ANALYSIS
# ==============================================================================
full_summary_path = REPO_ROOT / 'Models' / '6th try' / '2class' / 'evaluation_summary.json'
sliced_summary_path = REPO_ROOT / 'Models' / '6th try' / '2class' / 'Sliced_sensor_modals' / 'evaluation_summary.json'
eda_acc_summary_path = REPO_ROOT / 'Models' / '6th try' / '2class' / 'Minimum_sensor_modals' / 'evaluation_summary.json'

def extract_acc_map(summary_dict):
    acc_map = {}
    scheme_b = summary_dict.get('validation_scheme_b', {})
    if isinstance(scheme_b, dict):
        for k, v in scheme_b.items():
            clean = k.replace(', updated', '').replace(', Alkurdi', '').replace(' (Alkurdi)', '')
            acc_map[k] = v.get('pooled_accuracy')
            acc_map[clean] = v.get('pooled_accuracy')
    elif isinstance(scheme_b, list):
        for item in scheme_b:
            clean = item['Model'].replace(', updated', '').replace(', Alkurdi', '').replace(' (Alkurdi)', '')
            acc_map[item['Model']] = item.get('Pooled Accuracy')
            acc_map[clean] = item.get('Pooled Accuracy')
    return acc_map

def get_acc_map_from_path(p):
    if p.exists():
        with open(p, 'r', encoding='utf-8') as f:
            return extract_acc_map(json.load(f))
    return {}

full_acc_map = get_acc_map_from_path(full_summary_path)
sliced_acc_map = get_acc_map_from_path(sliced_summary_path)
eda_acc_map = get_acc_map_from_path(eda_acc_summary_path)

comparison_records = []
for _, row in loso_summary_df.iterrows():
    model_name = row['Model']
    min_acc = row['Pooled Accuracy']
    min_bal_acc = row['Pooled Balanced Acc']
    min_macro_f1 = row['Pooled Macro F1']

    def find_match(m_map):
        for k in [model_name, model_name.replace(' (max_depth=25)', ''), f'{model_name}, updated', f'{model_name}, Alkurdi']:
            if k in m_map:
                return m_map[k]
        return None

    full_acc = find_match(full_acc_map)
    sliced_acc = find_match(sliced_acc_map)
    eda_acc_val = find_match(eda_acc_map)

    shortfall_full = (full_acc - min_acc) if full_acc is not None else np.nan
    shortfall_sliced = (sliced_acc - min_acc) if sliced_acc is not None else np.nan
    retention_full = (min_acc / full_acc * 100) if full_acc is not None and full_acc > 0 else np.nan

    comparison_records.append({
        'Model': model_name,
        'Full (155f)': f'{full_acc*100:.2f}%' if full_acc is not None else 'N/A',
        'Sliced (62f)': f'{sliced_acc*100:.2f}%' if sliced_acc is not None else 'N/A',
        'EDA+ACC (56f)': f'{eda_acc_val*100:.2f}%' if eda_acc_val is not None else 'N/A',
        'EDA+TEMP (20f)': f'{min_acc*100:.2f}%',
        'Shortfall vs Full': f'{shortfall_full*100:+.2f}%' if not np.isnan(shortfall_full) else 'N/A',
        'Shortfall vs Sliced': f'{shortfall_sliced*100:+.2f}%' if not np.isnan(shortfall_sliced) else 'N/A',
        'Retention of Full': f'{retention_full:.2f}%' if not np.isnan(retention_full) else 'N/A',
        'ET Bal Acc': f'{min_bal_acc*100:.2f}%',
        'ET Macro F1': f'{min_macro_f1:.4f}'
    })

comp_df = pd.DataFrame(comparison_records)
print('=== Multi-Tier Shortfall: EDA+TEMP (20f) vs. EDA+ACC (56f) vs. Sliced (62f) vs. Full (155f) ===')
display(comp_df)


=== Multi-Tier Shortfall: EDA+TEMP (20f) vs. EDA+ACC (56f) vs. Sliced (62f) vs. Full (155f) ===


,Model,Full (155f),Sliced (62f),EDA+ACC (56f),EDA+TEMP (20f),Shortfall vs Full,Shortfall vs Sliced,Retention of Full,ET Bal Acc,ET Macro F1
0,Random Forest (max_depth=4),74.43%,71.14%,72.33%,67.99%,+6.44%,+3.15%,91.34%,68.12%,0.6780
1,Random Forest (max_depth=25),73.72%,69.87%,73.11%,69.50%,+4.22%,+0.37%,94.27%,69.92%,0.6940
2,XGBoost,74.14%,66.37%,68.69%,66.86%,+7.28%,-0.49%,90.18%,66.62%,0.6650
3,SVM (RBF),72.04%,63.91%,66.72%,69.21%,+2.83%,-5.30%,96.07%,69.68%,0.6912
4,KNN (k=5),67.64%,58.32%,62.88%,66.62%,+1.02%,-8.30%,98.49%,66.27%,0.6619
5,SVM Linear (C=1.0),64.96%,63.17%,65.47%,68.55%,-3.59%,-5.37%,105.52%,69.22%,0.6851
6,SVM Linear (C=0.9),64.98%,63.15%,65.45%,68.60%,-3.62%,-5.45%,105.56%,69.27%,0.6855
7,LDA,62.66%,67.62%,70.70%,71.18%,-8.52%,-3.57%,113.60%,72.46%,0.7118


### 8.2 Key Findings & Clinical/Wearable Feasibility Takeaways (EDA + TEMP):

1. **Autonomic-Only Feasibility (No Accelerometer):**
   - Evaluates state anxiety detection using purely autonomic markers without kinetic or motor artifact inputs.
   - Assesses whether electrodermal activity and peripheral skin temperature suffice for subject-independent classification.

2. **Validation Scheme Discrepancy:**
   - Validates that Scheme A continues to exhibit leakage inflation while Scheme B (LOSO CV) establishes real subject-independent generalization.